# Calibration and counterfactual fraud experiments

**Goal.** Fit a calibration profile against reference data, then run a controlled counterfactual intervention. The workflow keeps source truth immutable and records provenance for both artifacts.

**Audience.** Data/ML scientists, fraud analysts, platform engineers, and researchers.

**Prerequisites.** Python 3.12+, a clean checkout, and the base FraudTwin install. The workflow is deterministic and runs offline; service integrations are deliberately out of scope here.

**Source size.** 1,000–10,000 logical payments. Every section writes only compact summaries, manifests, or fingerprints to a temporary directory.

**Interpretation.** Synthetic evidence demonstrates mechanics and invariants, not production prevalence or model performance guarantees.


## Calibration against reference data



In [ ]:
from pathlib import Path

import polars as pl

from fraudtwin.calibration import (
    ReferenceDataset,
    apply_calibration_profile,
    fit_calibration_profile,
    load_calibration_profile,
    write_calibration_profile,
)
from fraudtwin.config import load_config
from fraudtwin.generation import generate
from fraudtwin.reproducibility import sha256_json

config = load_config(Path("configs/minimal.yaml")).model_copy(
    update={
        "payments": load_config(Path("configs/minimal.yaml")).payments.model_copy(
            update={"daily_target": 1000}
        )
    }
)
data = generate(config, write=False)
payments = data.behavior.payments
print({"run_id": data.run_id, "payments": len(payments)})
assert len(payments) >= 1000

In [ ]:
reference = pl.DataFrame(
    [
        {"amount": p.amount, "event_time": p.initiated_at, "customer_id": p.payer_account_id}
        for p in payments
    ]
)
print(reference.describe())
assert reference.height >= 1000

In [ ]:
ref = ReferenceDataset(
    reference,
    source_fingerprint=sha256_json(reference.to_dicts()),
    schema_fingerprint=sha256_json([(n, str(t)) for n, t in reference.schema.items()]),
)
profile = fit_calibration_profile(ref, seed=42)
print(
    {
        "profile_id": profile.profile_id,
        "summaries": len(profile.summaries),
        "distributions": len(profile.distributions),
    }
)

In [ ]:
summary_table = [
    {"name": s.name, "fields": s.fields, "available": s.available} for s in profile.summaries
]
print(pl.DataFrame(summary_table))
assert any(s["name"] == "amount_distribution" for s in summary_table)

In [ ]:
amount = next(d for d in profile.distributions if d.name == "amount")
print({"quantiles": amount.quantiles, "minimum": amount.minimum, "maximum": amount.maximum})
assert amount.minimum is not None and amount.maximum is not None

In [ ]:
shifted = reference.with_columns((pl.col("amount") * 1.15).alias("amount"))
print(
    {
        "reference_mean": round(reference["amount"].mean(), 2),
        "shifted_mean": round(shifted["amount"].mean(), 2),
    }
)
assert shifted["amount"].mean() > reference["amount"].mean()

In [ ]:
resolved = apply_calibration_profile(profile, seed=42)
print(
    {"enabled": resolved.enabled, "profile_id": resolved.profile_id, "streams": resolved.stream_ids}
)
assert resolved.enabled and resolved.profile_id == profile.profile_id

In [ ]:
from tempfile import TemporaryDirectory

with TemporaryDirectory() as tmp:
    path = write_calibration_profile(profile, Path(tmp) / "profile.yaml")
    loaded = load_calibration_profile(path)
print({"loaded": loaded.profile_id, "source_rows": loaded.provenance.row_count})
assert loaded.profile_id == profile.profile_id

In [ ]:
artifact = {
    "run_id": data.run_id,
    "profile_id": profile.profile_id,
    "source": profile.provenance.source_fingerprint,
    "rows": profile.provenance.row_count,
}
artifact["fingerprint"] = sha256_json(artifact)
print(artifact)

In [ ]:
assert artifact["fingerprint"] == sha256_json(
    {k: v for k, v in artifact.items() if k != "fingerprint"}
)
print("Calibration is versioned, deterministic, and append-only.")

## Counterfactual fraud experiment



In [ ]:
from pathlib import Path

from fraudtwin.config import load_config
from fraudtwin.generation import generate
from fraudtwin.reproducibility import sha256_json

base = load_config(Path("configs/benchmarks/m14-counterfactual-v1.yaml"))
config = base
data = generate(config, write=False)
source_ids = tuple(p.payment_id for p in data.behavior.payments)
source_fp = sha256_json(source_ids)
cf = data.behavior.counterfactual
print(
    {"run_id": data.run_id, "payments": len(source_ids), "counterfactual_id": cf.counterfactual_id}
)
assert cf is not None

In [ ]:
print(
    {
        "original": len(cf.original_payments),
        "modified": len(cf.modified_payments),
        "accepted": sum(x.status == "ACCEPTED" for x in cf.change_sets),
        "rejected": len(cf.rejected),
    }
)
assert cf.change_sets

In [ ]:
changes = [
    {
        "objective": x.objective,
        "status": x.status,
        "distance": x.effective_distance,
        "fields": tuple(x.changed_fields),
    }
    for x in cf.change_sets
]
print(changes)
assert changes

In [ ]:
accepted = [x for x in cf.change_sets if x.status == "ACCEPTED"]
print(
    {
        "accepted_objectives": [x.objective for x in accepted],
        "derived_ids": len(cf.modified_payments),
    }
)
assert all(x.derived_payment_id != x.source_payment_id for x in accepted)

In [ ]:
pairs = list(zip(cf.original_payments, cf.modified_payments, strict=False))
print({"paired": len(pairs), "amount_changes": sum(a.amount != b.amount for a, b in pairs)})

In [ ]:
from collections import Counter

counts = Counter(x.objective for x in cf.change_sets)
print(dict(counts))
assert sum(counts.values()) == len(cf.change_sets)

In [ ]:
source_after = sha256_json(tuple(p.payment_id for p in data.behavior.payments))
print({"before": source_fp, "after": source_after})
assert source_after == source_fp

In [ ]:
print(
    {"rejections": [{"objective": x.objective, "reason": x.rejection_reason} for x in cf.rejected]}
)

In [ ]:
manifest = {
    "counterfactual_id": cf.counterfactual_id,
    "source_run_id": data.run_id,
    "accepted": len(cf.modified_payments),
    "rejected": len(cf.rejected),
    "source_fingerprint": source_fp,
}
manifest["fingerprint"] = sha256_json(manifest)
print(manifest)

In [ ]:
assert manifest["fingerprint"] == sha256_json(
    {k: v for k, v in manifest.items() if k != "fingerprint"}
)
print("The intervention sidecar changes derived records only; source truth is unchanged.")

## Verification and next step

Re-run the offline cells from a clean checkout and compare the printed fingerprints. For service-backed publication, continue with the relevant integration runbook after this notebook; do not treat synthetic metrics as a deployment SLO.
